In [2]:
#carregando e ordenando dados
import pandas as pd 
import numpy as np

base = pd.read_parquet("../data/processed/dataset_modelagem.parquet")
base = base.sort_values("ini").reset_index(drop=True)
print(base.shape)

(4055, 65)


In [3]:
#entradas x e y
X_COLS = [
    "comprimento", "qtd_sessao", "tempo_prog_sessao_min", "quantidade_ordem",
    "troca_condutor", "troca_terminal_a", "troca_terminal_b", "troca_cor", "troca_ma_a", "troca_ma_b",
    "mesmo_sub_item", "mesma_programacao", "ocioso_antes_min", "lacuna_antes_6h",
    "hora_ini", "dia_semana", "ordem_no_dia", "n_subitens_kit",
    "n_ocorrencias_previas_sub", "dur_media_hist_sub", "razao_media_hist_sub",
    "setup_pcp", "maquina_diferente_programada",
]
base["mc_68"] = (base["mc"] == "68").astype(int)

X = base[X_COLS + ["mc_68"]].astype(float)
y = base["y_abastecimento"]

In [ ]:
#consolidando estatisticas de treino e teste para todos os modelos.
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold, cross_validate

modelos = {
    "Baseline":            DummyClassifier(strategy="most_frequent"),
    "Regressão Logística": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                         LogisticRegression(max_iter=1000)),
    "Árvore de Decisão":   make_pipeline(SimpleImputer(strategy="median"),
                                         DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=42)), #ajustar os parametros
    "Random Forest":       make_pipeline(SimpleImputer(strategy="median"),                                          #e econtrar os parametros ideais nos proximos passos
                                         RandomForestClassifier(n_estimators=300, min_samples_leaf=5, #usado valores padrão no scikitlearn
                                                                n_jobs=-1, random_state=42)),
    "Gradient Boosting":   HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                          min_samples_leaf=20, random_state=42),
}

metricas = ["accuracy", "precision", "recall", "f1", "roc_auc"]
validacoes = {
    "Temporal (5 janelas)": TimeSeriesSplit(n_splits=5),
    "K-fold embaralhado":   StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
}

linhas = []
for nome_val, cv in validacoes.items():
    for nome, modelo in modelos.items():
        r = cross_validate(modelo, X, y, cv=cv, scoring=metricas)   # treina e testa em cada janela
        linha = {"validação": nome_val, "modelo": nome}
        for m in metricas:
            linha[m] = r[f"test_{m}"].mean()                         # média das 5 janelas
        linha["auc_desvio"] = r["test_roc_auc"].std()                # quanto o resultado varia entre janelas
        linhas.append(linha)

resultados = pd.DataFrame(linhas)
resultados.round(3)

,validação,modelo,accuracy,precision,recall,f1,roc_auc,auc_desvio
0,Temporal (5 janelas),Baseline,0.720,0.720,1.000,0.837,0.500,0.000
1,Temporal (5 janelas),Regressão Logística,0.756,0.783,0.921,0.845,0.805,0.033
2,Temporal (5 janelas),Árvore de Decisão,0.780,0.835,0.865,0.850,0.792,0.040
3,Temporal (5 janelas),Random Forest,0.796,0.835,0.894,0.863,0.836,0.032
4,Temporal (5 janelas),Gradient Boosting,0.773,0.826,0.869,0.846,0.815,0.034
5,K-fold embaralhado,Baseline,0.723,0.723,1.000,0.839,0.500,0.000
6,K-fold embaralhado,Regressão Logística,0.782,0.811,0.911,0.858,0.818,0.016
7,K-fold embaralhado,Árvore de Decisão,0.779,0.834,0.867,0.850,0.812,0.014
8,K-fold embaralhado,Random Forest,0.798,0.831,0.905,0.866,0.844,0.016
9,K-fold embaralhado,Gradient Boosting,0.781,0.825,0.886,0.854,0.825,0.011
